# Figure-in-noise: the versions I tried

Meysam Amirsardari

Three task designs, how the stimulus is built and checked, and what one listener did.

[Open in Colab](https://colab.research.google.com/github/MeysamAmirsardari/SeqSFG_task/blob/main/notebooks/SeqSFG_overview.ipynb)

Run all. Headphones, fixed volume. Nothing autoplays. Code is hidden, click **Show code** to see any of it.

Pinned to `6a1bf7b00b69`. The pilot numbers come from the sessions in the repo, recorded with the
configuration you will hear.

In [ ]:
#@title setup
import sys, subprocess, importlib.util, json
from pathlib import Path
SOURCE_REF='6a1bf7b00b69b10b47f99453a8f7b8d8c419ef31'
REPO='https://github.com/MeysamAmirsardari/SeqSFG_task.git'
IN_COLAB=bool(importlib.util.find_spec('google') and importlib.util.find_spec('google.colab'))
if IN_COLAB:
    ROOT=Path('/content')/('seqsfg-'+SOURCE_REF[:12])
    if not ROOT.exists():
        subprocess.run(['git','clone','--no-checkout',REPO,str(ROOT)],check=True)
        subprocess.run(['git','-C',str(ROOT),'checkout','--detach',SOURCE_REF],check=True)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'seqsfg/stimulus.py').exists()),None)
for m in ['numpy','scipy','matplotlib']:
    if importlib.util.find_spec(m) is None: subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
sys.path.insert(0,str(ROOT))

import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Audio, Markdown
from seqsfg.config import Config, validate
from seqsfg.stimulus import make_trial, render_interval, FIGURE, BACKGROUND
from seqsfg.train import render_with_background_gain
from seqsfg import yesno, exposure as X, measure as Ms, pool as POOL

cfg=Config.from_dict(json.loads((ROOT/'pilot_config.json').read_text()))
D=validate(cfg); SR=cfg.sample_rate; OCT=np.log2(D.channel_freqs_hz)
SUM=json.loads((ROOT/'verification/pilot_summary.json').read_text())['sessions']
plt.rcParams.update({'figure.dpi':115,'axes.spines.top':False,'axes.spines.right':False,
                      'axes.grid':True,'grid.alpha':.25,'font.size':9})
C={'fig':'#c1272d','other':'#1b6ca8','bg':'#c3c8cf','ok':'#2e7d32','no':'#b0481f','ink':'#1f2328'}
TICKS=[250,1000,4000]

def play(x):
    pad=np.zeros(int(.05*SR),dtype=np.float32)
    return Audio(np.clip(np.concatenate([pad,x]),-1,1),rate=SR,normalize=False)

def part(iv,kind):
    m=iv.kind==kind; out=iv.copy()
    for a in ('onset','channel','phase','kind','element','component'): setattr(out,a,getattr(iv,a)[m])
    return render_interval(cfg,out,D)

def raster(ax,iv,col,t0=0,t1=None,title=''):
    t=iv.onset*cfg.grid_ms/1000.; y=OCT[iv.channel]; m=iv.kind==FIGURE
    ax.scatter(t[~m],y[~m],s=5,c=C['bg'],marker='s',lw=0)
    ax.scatter(t[m],y[m],s=22,c=col,marker='s',lw=0)
    ax.set_xlim(t0,t1 or cfg.interval_dur_ms/1000.); ax.set_xlabel('time (s)')
    ax.set_yticks(np.log2(TICKS)); ax.set_yticklabels([str(v) for v in TICKS]); ax.set_title(title,fontsize=10)

def cells_of(s): return SUM[s]['cells']
def dprime(h,ns,f,nn): return float(stats.norm.ppf((h+.5)/(ns+1))-stats.norm.ppf((f+.5)/(nn+1)))
def env(x): return Ms.frame_rms(x,SR,2.0)

print(f"{D.n_channels} channels, {cfg.pool_low_hz:.0f}-{D.channel_freqs_hz[-1]:.0f} Hz, one per ERB")
print(f"{cfg.n_elements} elements of {cfg.n_components} tones, {cfg.tone_dur_ms:.0f} ms each, "
      f"{1000/cfg.iei_max_ms:.1f}-{1000/cfg.iei_min_ms:.1f} per second, {cfg.interval_dur_ms/1000:g} s")
print(f"{cfg.tones_per_channel} tones in every channel, config {cfg.hash()}")

---
## 1 · The sound

A cloud of random tones. Somewhere in it, seven tones start together and come back on the same
pitches, nine times.

In [ ]:
#@title listen: cloud, figure, both
tr=make_trial(cfg,101,0.0,'rising',d=D); iv=tr.recurring
for lab,x in [('cloud only',part(iv,BACKGROUND)),('figure only',part(iv,FIGURE)),
              ('both, the real stimulus',render_interval(cfg,iv,D))]:
    display(Markdown(lab)); display(play(x))
fig,ax=plt.subplots(figsize=(11,3.2)); raster(ax,iv,C['fig'],title='red = the seven tones that start together')
ax.set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 2 · How it is built

Channels one ERB apart. Every channel gets the same number of tones, in both intervals, so
loudness and spectrum cannot give the answer away.

In [ ]:
#@title the channel pool and the figure band
S=tr.recurring.figure_set
fig,axes=plt.subplots(1,2,figsize=(12,2.9))
ax=axes[0]
ax.vlines(D.channel_freqs_hz,0,1,color=C['bg'],lw=1.4)
ax.vlines(D.channel_freqs_hz[S],0,1,color=C['fig'],lw=2.2)
ax.set_xscale('log'); ax.set_xticks([250,500,1000,2000,4000,8000]); ax.set_xticklabels(['250','500','1k','2k','4k','8k'])
ax.set_yticks([]); ax.set_xlabel('Hz'); ax.set_title('30 channels, red = this trial\'s figure',fontsize=10)
ax=axes[1]
gaps=np.diff(POOL.erb_number(D.channel_freqs_hz))
ax.plot(D.channel_freqs_hz[1:],gaps,'o-',color=C['ink'],ms=3,lw=1)
ax.set_xscale('log'); ax.set_ylim(0,1.6); ax.set_ylabel('spacing (ERB)'); ax.set_xlabel('Hz')
ax.set_xticks([250,1000,4000]); ax.set_xticklabels(['250','1k','4k'])
ax.set_title('equal spacing on the critical-band scale',fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
#@title the two intervals carry identical tones
a,b=tr.recurring,tr.other
ca=np.bincount(a.channel,minlength=D.n_channels); cb=np.bincount(b.channel,minlength=D.n_channels)
sa=Ms.channel_power_db(Ms.channel_envelopes(render_interval(cfg,a,D),SR,D.channel_freqs_hz,40.,10.))
sb=Ms.channel_power_db(Ms.channel_envelopes(render_interval(cfg,b,D),SR,D.channel_freqs_hz,40.,10.))
fig,axes=plt.subplots(1,2,figsize=(12,3.0))
ax=axes[0]; w=.4; ix=np.arange(D.n_channels)
ax.bar(ix-w/2,ca,w,color=C['fig'],label='A'); ax.bar(ix+w/2,cb,w,color=C['other'],label='B')
ax.set_xlabel('channel'); ax.set_ylabel('tones'); ax.legend(frameon=False,fontsize=8)
ax.set_title(f'tones per channel: identical, {cfg.tones_per_channel} everywhere',fontsize=10)
ax=axes[1]
ax.plot(D.channel_freqs_hz,sa-sa.mean(),color=C['fig'],lw=1.2,label='A')
ax.plot(D.channel_freqs_hz,sb-sb.mean(),color=C['other'],lw=1.2,ls='--',label='B')
ax.set_xscale('log'); ax.set_xticks([250,1000,4000]); ax.set_xticklabels(['250','1k','4k'])
ax.set_xlabel('Hz'); ax.set_ylabel('dB re mean'); ax.legend(frameon=False,fontsize=8)
ax.set_title(f'long-term spectrum, mean |A-B| = {np.abs(sa-sb).mean():.3f} dB',fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
#@title where the elements sit in time
x=render_interval(cfg,tr.recurring,D); e=env(x); t=np.arange(e.size)*2/1000.
fig,ax=plt.subplots(figsize=(11,2.8))
ax.plot(t,e/e.mean(),lw=.7,color=C['ink'])
for k,on in enumerate(tr.recurring.element_onsets*cfg.grid_ms/1000.):
    ax.axvline(on,color=C['fig'],lw=1,alpha=.55)
ax.set_xlabel('time (s)'); ax.set_ylabel('envelope / mean'); ax.set_xlim(0,cfg.interval_dur_ms/1000)
ax.set_title('red lines = the nine elements',fontsize=10); plt.tight_layout(); plt.show()

---
## 3 · Task A, two intervals

Two sounds. Both have a group every element. In one the group keeps the same pitches, in the
other it moves. Which one repeats?

In [ ]:
#@title listen: A and B
for lab,ivx in [('interval A',tr.recurring),('interval B',tr.other)]:
    display(Markdown(lab)); display(play(render_interval(cfg,ivx,D)))
fig,axes=plt.subplots(1,2,figsize=(12,3.4),sharey=True)
raster(axes[0],tr.recurring,C['fig'],title='A: same pitches every time')
raster(axes[1],tr.other,C['other'],title='B: new pitches every time')
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 4 · The knob: how much the tones overlap

Inside an element the seven tones can start together or spread out. `step` is the gap between
neighbours.

In [ ]:
#@title what one element looks like at each step
fig,axes=plt.subplots(len(cfg.steps_ms),2,figsize=(12,10),sharex=True)
for row,s in enumerate(cfg.steps_ms):
    for col,v in enumerate(('rising','redrawn')):
        t=make_trial(cfg,3,s,v,d=D).recurring
        k=1; sel=np.flatnonzero((t.element==k)&(t.kind==FIGURE))
        t0=t.onset[sel].min()*cfg.grid_ms
        ax=axes[row,col]
        for j in sel:
            st=t.onset[j]*cfg.grid_ms-t0
            ax.barh(int(t.component[j]),cfg.tone_dur_ms,left=st,height=.45,color='#127a7f')
        ax.set_ylim(-.7,cfg.n_components-.3); ax.set_yticks(range(cfg.n_components))
        ax.set_title(f'{v}: {s:.0f} ms',fontsize=9)
        if row==len(cfg.steps_ms)-1: ax.set_xlabel('time from the first tone (ms)')
        if col==0: ax.set_ylabel('component')
plt.tight_layout(); plt.show()

In [ ]:
#@title overlap, span and how many sound at once
steps=np.array(cfg.steps_ms)
ov=np.clip(1-steps/cfg.tone_dur_ms,0,1)
span=(cfg.n_components-1)*steps+cfg.tone_dur_ms
sim=[min(cfg.n_components,int(np.ceil(cfg.tone_dur_ms/s))) if s>0 else cfg.n_components for s in steps]
fig,axes=plt.subplots(1,3,figsize=(12,2.8))
for ax,y,lab in zip(axes,[ov,span,sim],['overlap','element span (ms)','tones sounding at once']):
    ax.plot(steps,y,'o-',color=C['ink'],lw=1.3); ax.set_xlabel('step (ms)'); ax.set_title(lab,fontsize=10)
axes[0].set_ylim(0,1.05); plt.tight_layout(); plt.show()
for s in (0.0,17.0):
    display(Markdown(f'step {s:.0f} ms, overlap {np.clip(1-s/cfg.tone_dur_ms,0,1):.2f}'))
    display(play(render_interval(cfg,make_trial(cfg,7,s,'rising',d=D).recurring,D)))

---
## 5 · The same figure at every step

Same seven pitches, same element, only the gap between onsets changes. Left alone, right in the
cloud.

In [ ]:
#@title one element at each step, alone and in the cloud
STEPS=list(cfg.steps_ms); SEED=23; ELEM=2
fig,axes=plt.subplots(len(STEPS),2,figsize=(12,1.95*len(STEPS)),sharey=True)
for r,s in enumerate(STEPS):
    iv=make_trial(cfg,SEED,s,'rising',d=D).recurring
    sel=np.flatnonzero((iv.element==ELEM)&(iv.kind==FIGURE))
    t0=iv.onset[sel].min()*cfg.grid_ms
    lo,hi=-50.,(cfg.n_components-1)*s+cfg.tone_dur_ms+50.
    for col,(ttl,show_bg) in enumerate([('alone',False),('in the cloud',True)]):
        ax=axes[r,col]
        if show_bg:
            for j in np.flatnonzero(iv.kind==BACKGROUND):
                st=iv.onset[j]*cfg.grid_ms-t0
                if st>hi or st+cfg.tone_dur_ms<lo: continue
                ax.plot([st,st+cfg.tone_dur_ms],[OCT[iv.channel[j]]]*2,
                        color=C['bg'],lw=2.8,solid_capstyle='butt')
        for j in sel:
            st=iv.onset[j]*cfg.grid_ms-t0
            ax.plot([st,st+cfg.tone_dur_ms],[OCT[iv.channel[j]]]*2,
                    color=C['fig'],lw=3.2,solid_capstyle='butt')
        ax.set_xlim(lo,hi); ax.grid(alpha=.25)
        ax.set_yticks(np.log2(TICKS)); ax.set_yticklabels([str(v) for v in TICKS])
        if r==0: ax.set_title(ttl,fontsize=11)
        if r==len(STEPS)-1: ax.set_xlabel('time from the first tone of the element (ms)')
    axes[r,0].set_ylabel(f'{s:.0f} ms\noverlap {max(0,1-s/cfg.tone_dur_ms):.2f}',fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
#@title listen: each step, figure alone then the same trial in the cloud
for s in STEPS:
    iv=make_trial(cfg,SEED,s,'rising',d=D).recurring
    display(Markdown(f"**{s:.0f} ms**, overlap {max(0,1-s/cfg.tone_dur_ms):.2f}"))
    display(Markdown('figure alone')); display(play(part(iv,FIGURE)))
    display(Markdown('in the cloud')); display(play(render_interval(cfg,iv,D)))

At 0 ms it is a chord. By 17 ms the seven tones are a ramp almost as long as the gap to the next
element, and in the right hand column it stops looking like one thing.

---
## 6 · Turning the background down

The training pilot walked the background up in steps. Same figure, same timing, only the cloud
changes. This is how I check people can hear the target at all before testing them.

In [ ]:
#@title listen: the background ladder, -30 dB up to the real thing
LV=[-30.,-24.,-18.,-12.,-6.,0.]
base=make_trial(cfg,17,7.0,'rising',d=D).recurring
for g in LV:
    display(Markdown(f"background {g:+.0f} dB" + ("   <- the real stimulus" if g==0 else "")))
    display(play(render_with_background_gain(cfg,D,base,g)))

In [ ]:
#@title what the ladder does to the sound
fig,axes=plt.subplots(len(LV),1,figsize=(11,7),sharex=True,sharey=True)
for ax,g in zip(axes,LV):
    x=render_with_background_gain(cfg,D,base,g); e=env(x)
    t=np.arange(e.size)*2/1000.
    ax.plot(t,e/env(render_with_background_gain(cfg,D,base,0.)).mean(),lw=.7,color=C['ink'])
    for on in base.element_onsets*cfg.grid_ms/1000.: ax.axvline(on,color=C['fig'],lw=.9,alpha=.5)
    ax.set_ylabel(f'{g:+.0f} dB',fontsize=9); ax.set_xlim(.3,2.2)
axes[-1].set_xlabel('time (s)')
axes[0].set_title('envelope at each background level, red = elements',fontsize=10)
plt.tight_layout(); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,3.0))
snr=[]; peak=[]
for g in LV:
    f=part(base,FIGURE); b=part(base,BACKGROUND)*10**(g/20)
    snr.append(20*np.log10(np.sqrt(np.mean(f**2))/np.sqrt(np.mean(b**2))))
    x=render_with_background_gain(cfg,D,base,g); e=env(x); peak.append(e.max()/e.mean())
axes[0].plot(LV,snr,'o-',color=C['fig'],lw=1.3); axes[0].axhline(0,color='k',lw=.8,ls='--')
axes[0].set_xlabel('background gain (dB)'); axes[0].set_ylabel('figure minus cloud (dB)')
axes[0].set_title('how far the figure sticks out',fontsize=10)
axes[1].plot(LV,peak,'o-',color=C['ink'],lw=1.3)
axes[1].set_xlabel('background gain (dB)'); axes[1].set_ylabel('peak / mean')
axes[1].set_title('envelope crest',fontsize=10)
plt.tight_layout(); plt.show()

At −30 dB the figure is obvious. At 0 dB it is the real task. Everyone I ran could do the top of
this ladder and nobody was reliable at the bottom.

---
## 7 · Task A, one listener

120 trials, 10 per point.

In [ ]:
#@title percent correct against step
cc=cells_of('L3/session_06')
fig,ax=plt.subplots(figsize=(6.6,3.6))
for v,col in (('rising',C['fig']),('redrawn',C['other'])):
    xs,ys,lo,hi=[],[],[],[]
    for key in sorted([k for k in cc if k.startswith(v+'|')],key=lambda k:float(k.split('|')[1])):
        n,kk=cc[key]['n'],cc[key]['n_correct']; p=kk/n
        a,b=stats.beta.interval(.95,kk+.5,n-kk+.5)
        xs.append(float(key.split('|')[1])); ys.append(p); lo.append(max(0,p-a)); hi.append(max(0,b-p))
    ax.errorbar(xs,ys,yerr=[lo,hi],fmt='o-',color=col,capsize=3,lw=1.4,label=v)
ax.axhline(.5,color='k',lw=.8,ls='--'); ax.text(17,.52,'chance',fontsize=8,ha='right')
ax.set_xlabel('step (ms)'); ax.set_ylabel('correct'); ax.set_ylim(0,1.05); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

Both curves: perfect when the tones start together, chance as soon as they do not.

So the task is measuring simultaneity, not whether the pitches repeat. The two orders lie on top
of each other, so there is no order effect to find.

---
## 8 · All the comparisons I tried

Six versions of "what is the other sound". The first one is the easy version I started with. The
middle three are the same question with the timing inside the element changed. The last two are
controls that should be impossible.

| | interval A | interval B |
|---|---|---|
| ungrouped | figure repeating | no group at all |
| rising | figure repeating, low to high | a group on new pitches each time |
| scrambled | same, one fixed scrambled order | same |
| redrawn | same, new order every element | same |
| onechannel | one pitch coming back, no group | nothing coming back |
| scattered | pitches come back but never line up | new pitches, also scattered |

In [ ]:
#@title listen: all six
VS=[('ungrouped','figure vs nothing'),('rising','figure vs different pitches, low to high'),
    ('scrambled','same, one fixed order'),('redrawn','same, new order every element'),
    ('onechannel','one pitch recurring, no group'),('scattered','pitches recur, never lined up')]
fig,axes=plt.subplots(3,2,figsize=(12,8),sharey=True)
for ax,(v,ttl) in zip(axes.ravel(),VS):
    raster(ax,make_trial(cfg,55,14.0,v,d=D).recurring,C['fig'],t0=.3,t1=1.7,title=f'{v}: {ttl}')
for ax in axes[:,0]: ax.set_ylabel('Hz')
plt.tight_layout(); plt.show()
for v,ttl in VS:
    t2=make_trial(cfg,55,14.0,v,d=D)
    display(Markdown(f'**{v}** - {ttl}'))
    display(play(np.concatenate([render_interval(cfg,t2.recurring,D),
                                 np.zeros(int(.35*SR),dtype=np.float32),
                                 render_interval(cfg,t2.other,D)])))

Each player is A, a gap, then B.

ungrouped is the one people can do. Everything below it is the real question and much harder.

In [ ]:
#@title how much of each interval is a group
rowsv=[]
for v,_ in VS:
    t2=make_trial(cfg,55,14.0,v,d=D)
    na=int((t2.recurring.kind==FIGURE).sum()); nb=int((t2.other.kind==FIGURE).sum())
    sets=len({tuple(x.tolist()) for x in t2.other.element_sets})
    rowsv.append((v,na,nb,sets))
fig,axes=plt.subplots(1,2,figsize=(12,3.0))
lab=[r[0] for r in rowsv]; x=np.arange(len(lab)); w=.38
axes[0].bar(x-w/2,[r[1] for r in rowsv],w,color=C['fig'],label='A')
axes[0].bar(x+w/2,[r[2] for r in rowsv],w,color=C['other'],label='B')
axes[0].set_xticks(x); axes[0].set_xticklabels(lab,rotation=20,ha='right')
axes[0].set_ylabel('tones in a group'); axes[0].legend(frameon=False,fontsize=8)
axes[0].set_title('both sides grouped, except the easy version',fontsize=10)
axes[1].bar(x,[r[3] for r in rowsv],color=C['bg'],edgecolor='k',lw=.6)
axes[1].set_xticks(x); axes[1].set_xticklabels(lab,rotation=20,ha='right')
axes[1].set_ylabel('distinct pitch sets in B'); axes[1].set_title('1 would mean B repeats too',fontsize=10)
plt.tight_layout(); plt.show()

---
## 9 · Task B, one interval

One sound. Was a figure there, yes or no. Easier to follow than comparing two.

In [ ]:
#@title listen: yes and no
for present in (True,False):
    display(Markdown(f'**{"yes" if present else "no"}**'))
    display(play(render_interval(cfg,yesno.build_interval(cfg,D,31 if present else 32,7.0,present,'roving'),D)))
fig,axes=plt.subplots(1,2,figsize=(12,3.4),sharey=True)
raster(axes[0],yesno.build_interval(cfg,D,31,7.0,True,'roving'),C['fig'],title='yes: the group returns')
raster(axes[1],yesno.build_interval(cfg,D,32,7.0,False,'roving'),C['other'],title='no: a group, but never the same')
axes[0].set_ylabel('Hz'); plt.tight_layout(); plt.show()

---
## 10 · What "no figure" should be

Three options. I checked whether a machine that never hears a group can separate each from a
"yes" trial, using 65 acoustic measures.

In [ ]:
#@title listen: three versions of no, and how separable each is
for k,lab in [('plain','plain cloud, no group at all'),('scattered','same pitches, never lined up'),
              ('roving','a group, but new pitches each time')]:
    display(Markdown(f'`{k}` - {lab}')); display(play(render_interval(cfg,yesno.build_interval(cfg,D,88,7.0,False,k),D)))
res={}
for k in ('plain','scattered','roving'):
    t=(ROOT/f'verification/yesno_{k}_report.txt').read_text()
    res[k]=float([l for l in t.splitlines() if 'hit rate' in l and 'correct' in l][0].split()[-2].rstrip('%'))
fig,axes=plt.subplots(1,2,figsize=(12,3.0))
ax=axes[0]
ax.bar(list(res),[res[k] for k in res],color=[C['no'] if k!='roving' else C['ok'] for k in res])
ax.axhline(50,color='k',lw=.8,ls='--'); ax.set_ylabel('% correct, machine'); ax.set_ylim(0,100)
for i,k in enumerate(res): ax.text(i,res[k]+2,f'{res[k]:.0f}%',ha='center',fontsize=9)
ax.set_title('can a machine do it without hearing a group?',fontsize=10)
ax=axes[1]
for k,col in (('plain',C['no']),('roving',C['ok'])):
    x=render_interval(cfg,yesno.build_interval(cfg,D,88,7.0,False,k),D)
    e=env(x); dev=e/e.mean()-1; ac=np.correlate(dev,dev,'full')[dev.size-1:]; ac/=ac[0]
    ax.plot(np.arange(600)*2,ac[:600],lw=1.2,color=col,label=k)
x=render_interval(cfg,yesno.build_interval(cfg,D,88,7.0,True,'roving'),D)
e=env(x); dev=e/e.mean()-1; ac=np.correlate(dev,dev,'full')[dev.size-1:]; ac/=ac[0]
ax.plot(np.arange(600)*2,ac[:600],lw=1.2,color=C['fig'],label='yes')
ax.axvspan(cfg.iei_min_ms,cfg.iei_max_ms,color='k',alpha=.07)
ax.set_xlabel('lag (ms)'); ax.set_ylabel('envelope autocorrelation'); ax.legend(frameon=False,fontsize=8)
ax.set_title('a chord every 316 ms is a rhythm',fontsize=10)
plt.tight_layout(); plt.show()

A plain cloud is 90% solvable from the envelope alone. So "no" has to mean a group that does not
repeat.

---
## 11 · Task B, one listener

120 trials, 10 present and 10 absent per point.

In [ ]:
#@title hit rate, false alarms and d' against step
c7=cells_of('L3/session_07')
xs=[float(k.rstrip('|')) for k in sorted(c7,key=lambda k:float(k.rstrip('|')))]
hit=[c7[k]['hits']/c7[k]['n_signal'] for k in sorted(c7,key=lambda k:float(k.rstrip('|')))]
fa=[c7[k]['false_alarms']/c7[k]['n_noise'] for k in sorted(c7,key=lambda k:float(k.rstrip('|')))]
dp=[dprime(c7[k]['hits'],c7[k]['n_signal'],c7[k]['false_alarms'],c7[k]['n_noise'])
    for k in sorted(c7,key=lambda k:float(k.rstrip('|')))]
fig,axes=plt.subplots(1,2,figsize=(12,3.3))
axes[0].plot(xs,hit,'o-',color=C['fig'],lw=1.4,label='hits')
axes[0].plot(xs,fa,'s--',color=C['other'],lw=1.4,label='false alarms')
axes[0].set_ylim(0,1.05); axes[0].set_xlabel('step (ms)'); axes[0].set_ylabel('rate'); axes[0].legend(frameon=False)
axes[1].plot(xs,dp,'o-',color=C['ink'],lw=1.4); axes[1].axhline(0,color='k',lw=.8,ls='--')
axes[1].set_xlabel('step (ms)'); axes[1].set_ylabel("d'")
plt.tight_layout(); plt.show()

Same shape. Something at step 0, nothing after. The false alarms climb, which is the part that
worries me most.

---
## 12 · Task C, training inside the task

Two orders, P and Q, on the same seven pitches, chosen to share no transition.

Test both, give extra trials of one only, test both again.

In [ ]:
#@title the two orders, and what they share
P,Q,OV=X.choose_orders(cfg.n_components,20260910)
S2=X.figure_set_for(cfg,D,X.ExposureConfig())
fig,axes=plt.subplots(1,2,figsize=(12,3.0))
ax=axes[0]
for k,o,col,off in (('P',P,C['fig'],-.16),('Q',Q,C['other'],.16)):
    ax.barh(np.arange(cfg.n_components)+off,cfg.tone_dur_ms,left=np.asarray(o)*14.,height=.3,color=col,label=k)
ax.set_yticks(range(cfg.n_components)); ax.set_ylabel('component'); ax.set_xlabel('onset (ms), step 14')
ax.legend(frameon=False,fontsize=8); ax.set_title('same pitches, different timing',fontsize=10)
ax=axes[1]
lab=['shared\ntransitions','shared\nadjacencies','same\nslot']
val=[OV['shared_directed_transitions'],OV['shared_undirected_adjacencies'],OV['components_in_the_same_slot']]
ax.bar(lab,val,color=C['bg'],edgecolor='k',lw=.6); ax.set_ylim(0,7)
ax.set_ylabel(f'out of {cfg.n_components}'); ax.set_title('what P and Q still share',fontsize=10)
plt.tight_layout(); plt.show()
for k,o in (('P',P),('Q',Q)):
    display(Markdown(f'order {k}')); display(play(render_interval(cfg,X.build_trial(cfg,D,S2,o,64,14.0,True),D)))

The effect:

D = (post trained − pre trained) − (post untrained − pre untrained)

Subtracting the untrained side removes plain practice.

In [ ]:
#@title the exposure pilot
c8=cells_of('L3/session_08'); dd={}
fig,axes=plt.subplots(1,2,figsize=(12,3.5))
ax=axes[0]
for role,col in (('trained',C['fig']),('untrained',C['other'])):
    for phase,ls,mk in (('pre','--','o'),('post','-','s')):
        xs,ys=[],[]
        for key in sorted([k for k in c8 if k.startswith(f'{phase}|{role}|')],key=lambda k:float(k.split('|')[2])):
            v=c8[key]; st=float(key.split('|')[2])
            g=dprime(v['hits'],v['n_signal'],v['false_alarms'],v['n_noise'])
            dd[(phase,role,st)]=g; xs.append(st); ys.append(g)
        ax.plot(xs,ys,ls,marker=mk,color=col,lw=1.3,ms=5,label=f'{role} {phase}')
ax.axhline(0,color='k',lw=.8,ls='--'); ax.set_xlabel('step (ms)'); ax.set_ylabel("d'")
ax.legend(frameon=False,fontsize=8,ncol=2); ax.set_title('four curves',fontsize=10)
ax=axes[1]
sts=sorted({k[2] for k in dd if k[2]>0})
Dv=[(dd[('post','trained',s)]-dd[('pre','trained',s)])-(dd[('post','untrained',s)]-dd[('pre','untrained',s)]) for s in sts]
ax.bar([f'{s:.0f} ms' for s in sts],Dv,color=C['bg'],edgecolor='k',lw=.6)
ax.errorbar([f'{s:.0f} ms' for s in sts],Dv,yerr=[2.0]*len(sts),fmt='none',ecolor='k',capsize=4)
ax.axhline(0,color='k',lw=.8); ax.set_ylabel('D'); ax.set_ylim(-4,4)
ax.set_title('D, with roughly how wide 8 trials a cell makes it',fontsize=10)
plt.tight_layout(); plt.show()

D is flat and the bars are about as wide as the plot. One listener, 8 trials a cell. Not enough
to say anything.

---
## 13 · The versions before this one

Every session starts with practice: you have to get 10 of 12 before the real block begins. That
is a good filter, and for a long time nothing passed it.

In [ ]:
#@title practice scores across all the versions recorded
order=[k for k in SUM if SUM[k].get('practice')]
def frac(k):
    p=SUM[k]['practice']; n=sum(v['n'] for v in p.values()); c=sum(v['n_correct'] for v in p.values())
    return c,n
lab,val,cols,note=[],[],[],[]
for k in order:
    c,n=frac(k); st=SUM[k]['settings']
    lab.append(k); val.append(c/n)
    cols.append(C['ok'] if SUM[k]['status']=='complete' else C['no'])
    note.append(f"{st['tone_dur_ms']:.0f} ms, {st['interval_dur_ms']/1000:g} s, "
                f"K={st['n_elements']}, M={st['tones_per_channel']}, "
                f"band={st['figure_band_channels'] or '-'}")
fig,ax=plt.subplots(figsize=(11,4.2))
y=np.arange(len(lab))
ax.barh(y,val,color=cols)
ax.axvline(10/12,color='k',ls='--',lw=1); ax.text(10/12,-.9,'criterion 10 of 12',fontsize=8,ha='center')
ax.set_yticks(y); ax.set_yticklabels([f'{l}   {nn}' for l,nn in zip(lab,note)],fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0,1.02); ax.set_xlabel('practice correct')
ax.set_title('green = went on to a full block',fontsize=10)
plt.tight_layout(); plt.show()
for k in order:
    c,n=frac(k); print(f"{k}  {c}/{n}  {SUM[k]['status']}")

Reading down: 30 ms tones in a 2.25 s cloud, then 60 ms in 3 s, then 45 ms in 4 s. The first one
that got past practice is the one with the figure squeezed into a band of 13 channels, which is
also the first one where the two sounds are obviously different pitches rather than two spreads
of the same five octaves.

So the stimulus work did move something. It is the step curve that has not.

---
## 14 · The check I run on every version

Ideal observers that get one property of the sound and nothing else. They have to sit at chance,
or the task can be done without hearing a figure.

In [ ]:
#@title blind observers on the two-interval task
import re
t=(ROOT/'verification/pilot_battery_report.txt').read_text().splitlines()
i=[n for n,l in enumerate(t) if "ladder 'rising'" in l][0]
names,vals,ci=[],[],[]
for l in t[i+1:i+6]:
    m=re.match(r"\s*(.+?)\s+d' = ([+-][0-9.]+)\s+\[([+-][0-9.]+), ([+-][0-9.]+)\]",l)
    if m:
        names.append(m.group(1)); vals.append(float(m.group(2)))
        ci.append((float(m.group(2))-float(m.group(3)),float(m.group(4))-float(m.group(2))))
fig,ax=plt.subplots(figsize=(7,2.9))
ax.barh(names[::-1],vals[::-1],xerr=np.array(ci[::-1]).T,color=C['bg'],edgecolor='k',lw=.6,
        error_kw=dict(ecolor='k',capsize=3))
ax.axvline(0,color='k',lw=.9); ax.set_xlim(-1,1); ax.set_xlabel("d'  (0 = chance)")
plt.tight_layout(); plt.show()
print([l for l in t[i:i+12] if 'global permutation' in l][0].strip())

---
## 15 · Where it is

Works: the stimulus is controlled, the machine checks pass, three tasks run end to end.

Does not work yet: past step 0 the listener is at chance, so there is no curve to fit and no
order effect to look for.

Next: more trials per point, and start from an easier setting so the curve begins above chance.
The background ladder in section 5 is the obvious place to take that from.